# 商品实体匹配：数据、特征与评价

[🌐 English](product-matching-walkthrough.ipynb) | **🇨🇳 简体中文**

同一商品在两家零售商处，可能使用不同的标题。这个实验利用文本和字段比较来判断两条记录是否对应。先看已保存的实验结果，再用几个小例子理解输入处理和评价指标。

最终模型在本项目中简称 **C+**：RoBERTa 读取商品对文本，并融合 42 项字段比较，再平均三个训练种子的匹配概率。后面的对照表会分别说明各个方法。

真实结果来自 **555 对固定的 Walmart–Amazon 商品记录**。这是回顾性比较：历史接触记录不完整，Walmart–Amazon 独立新样本的表现尚未验证。下文的 Acme 商品、标签和概率均为教学编写，不是模型预测。

**阅读或运行**：GitHub 会直接显示保存输出，阅读讲解无需配置环境。若要交互运行，使用已有的 Jupyter Python 3.11 及以上内核，按顺序运行单元。也可以按照 [README 快速开始](../README.zh-CN.md#快速开始)创建轻量项目环境，使用其中显式的解释器运行 `scripts/run_notebook.py`：它仅用标准库执行中英文两份讲解并核对保存输出，无需安装额外包。这些单元不需要商品数据集或下载模型。

[项目概览](../README.zh-CN.md) · [数据与结果](../showcase/zh-CN/index.html) · [复现指南](../docs/zh-CN/REPRODUCIBILITY.md)


In [1]:
from pathlib import Path
import json, math, sys
LANGUAGE = "zh-CN"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "results/fixed-results.json").is_file() and (p / "src/product_matching").is_dir())
sys.path.insert(0, str(ROOT / "src"))
facts = json.loads((ROOT / "results/fixed-results.json").read_text(encoding="utf-8"))
text = json.loads((ROOT / "notebooks/walkthrough_text.json").read_text(encoding="utf-8"))[LANGUAGE]
print(text["task"])
for role, count in facts["roles"].items():
    print(f'{text["roles"][role]}: {count:,}')


任务：判断两家零售商的商品记录是否指向同一商品。
训练商品对: 3,738
开发商品对: 199
校准商品对: 209
评价商品对: 555


## 1. 数据用途与标签分离

| 用途 | 商品对数量 | 做什么 |
| --- | ---: | --- |
| 训练 | 3,738 | 更新模型参数，拟合标题 IDF |
| 开发 | 199 | 选择共同检查点轮数 |
| 校准 | 209 | 选择分类阈值 |
| 评价 | 555 | 衡量已冻结的模型与阈值 |

评价池中有 **190 对匹配、365 对不匹配**。IDF（逆文档频率）提高训练标题中较少出现词语的权重，降低常见词对标题相似度的影响。

预测与标签通过 `pair_id` 对齐，不依赖行顺序。只有**训练用途的模型输入**携带目标标签；其他用途的标签与预测输入分开，在选择检查点、校准阈值或计算指标时再对齐。

数据流是：`训练集拟合 IDF → 按用途准备输入 → 开发集选轮数 → 校准集选阈值 → 评价`。这些用途说明预定使用方式，不能证明历史评价样本从未被接触。


## 2. 固定评价池的模型对照

六种方法使用同一批 555 对评价样本。各方法的名称对应以下模型和比较特征：

| 方法 | 说明 |
| --- | --- |
| B22 | 使用 22 项比较特征的逻辑回归 |
| L30 | 使用 30 项比较特征的直方图梯度提升树 |
| S33 | L30 加三项固定的 MiniLM 语义／缺失状态特征 |
| L42 | 使用全部 42 项比较特征的直方图梯度提升树 |
| C+ | RoBERTa 融合 42 项比较特征，平均三个训练种子的概率 |
| C0 | 独立训练的神经模型对照，训练和推断时均将 42 项特征输入置零 |

下一单元读取**已保存的聚合结果**。**AP**（平均精确率）反映正例在排序中的位置；**F1** 在各方法的校准阈值下综合考虑误判与漏判；**P@100** 是排序最前 100 对的精确率。AP 不等于分类准确率。


In [2]:
print(f'{text["method"]:<8} {"AP":>9} {"F1":>9} {"P@100":>9}')
for method, result in facts["metrics"].items():
    print(f'{method:<8} {result["ap"]:9.6f} {result["f1"]:9.6f} {result["p_at_100"]:9.2f}')


方法              AP        F1     P@100
B22       0.795590  0.743719      0.81
L30       0.844899  0.762115      0.90
S33       0.846702  0.767123      0.89
L42       0.949700  0.852041      1.00
C+        0.965998  0.901639      1.00
C0        0.965616  0.875648      1.00


### 指标绝对差与误差计数

L42 是这批样本中表现最好的表格模型对照。将 C+ 的分数减去 L42，再乘以 100，就得到以百分点表示的绝对差值。同时比较独立训练的 C0。已保存的误判与漏判计数，有助于解释各方法在自身校准阈值下的变化。


In [3]:
cplus, reference, ablation = (facts["metrics"][m] for m in ("C+", "L42", "C0"))
print(f'{text["ap_gain"]}: {(cplus["ap"]-reference["ap"])*100:.4f} {text["pp"]}')
print(f'{text["f1_gain"]}: {(cplus["f1"]-reference["f1"])*100:.4f} {text["pp"]}')
print(f'{text["ap_c0"]}: {(cplus["ap"]-ablation["ap"])*100:.4f} {text["pp"]}')
print(f'{text["f1_c0"]}: {(cplus["f1"]-ablation["f1"])*100:.4f} {text["pp"]}')
for method, result in (("L42", reference), ("C+", cplus)):
    counts = result["confusion"]
    print(f'{method} {text["error_counts"]}: FP={counts["fp"]}, FN={counts["fn"]}')


相较 L42 的 AP 增量: 1.6298 个百分点
相较 L42 的 F1 增量: 4.9599 个百分点
相较 C0 的 AP 差值: 0.0381 个百分点
各自校准阈值下，相较 C0 的 F1 差值: 2.5992 个百分点
L42 已保存的误差计数: FP=35, FN=23
C+ 已保存的误差计数: FP=11, FN=25


在各自的校准阈值下，相比 L42，C+ 将误判为匹配的数量从 **35 降至 11**，漏掉的匹配则从 **23 增至 25**。精确率提高，召回率略降；F1 更高并不意味着两类误差都减少。L42、C+ 和 C0 的 P@100 均为 1.00，因此这个指标无法区分它们在这批样本中排名最前的 100 对。

C+ 与 C0 的 **AP 只差约 0.04 个百分点**，C+ 的 **F1 则在各自单独校准的阈值下高约 2.60 个百分点**。两者独立训练，这组结果不能证明 42 项特征带来了明显的 AP 增益，也不能证明差异具有统计显著性。比较描述的是这批固定样本，独立新样本的表现仍未验证。


## 3. 合成商品对的特征构造

两条 Acme 记录是编写的例子，型号标点、标题空格和价格有所不同。另两个变体分别将右侧型号从 `ZX100` 改为 `ZX101`，或将币种从 USD 改为 EUR。下表使用**同一个特征函数和本例已拟合的标题 IDF 状态**比较三种输入。

42 项特征由 **14 项业务字段比较 + 8 项标题比较（4 项训练数据拟合的 IDF 加权比较、4 项直接比较）+ 8 项标题型号代码比较 + 12 项原生型号与类别比较**组成。业务字段包括标题、品牌、描述、价格和币种；原生字段是来源记录提供的 `modelno` 和 `category`，与业务字段视图分开保留。全部 42 项标识见[数据卡与特征字典](../docs/zh-CN/DATA_CARD.md)。

表中展示的是二元标志：1 表示满足列名所述条件，0 表示不满足。紧凑标题和紧凑型号比较会忽略空格与标点。不同型号的那行还标出**型号字母相同、数字不同**；EUR 那行则使 USD／EUR 的价格比较不可用。缺失价格保持缺失，不会被当成观测到的零价格。

这里编写的训练标签只用于演示输入结构与 IDF 构建。**没有训练分类器，RoBERTa 也没有为这些记录打分。**字段比较提供证据，不是最终匹配判定。特征代码的位置见[代码导览](../docs/zh-CN/CODE_MAP.md)。


In [4]:
from product_matching.data import business_record, native_record
from product_matching.features import FEATURE_NAMES_42, fit_title_evidence, prepare_rows
raw_left = {"title":"Acme ZX-100 16GB", "brand":"Acme", "price":"19.99", "priceCurrency":"USD"}
raw_right = {"title":"ACME ZX100 16 GB", "brand":"ACME", "price":"20.00", "priceCurrency":"USD"}
example = {"pair_id":"synthetic:acme", "left":business_record(raw_left), "right":business_record(raw_right),
           "left_native":native_record({"modelno":"ZX-100", "category":"electronics"}),
           "right_native":native_record({"modelno":"ZX100", "category":"electronics"})}
fit_idf = fit_title_evidence([{**example, "label":1}])
different_model = {
    **example, "pair_id":"synthetic:acme-different-model",
    "right":business_record({**raw_right, "title":"ACME ZX101 16 GB"}),
    "right_native":native_record({"modelno":"ZX101", "category":"electronics"}),
}
different_currency = {
    **example, "pair_id":"synthetic:acme-different-currency",
    "right":business_record({**raw_right, "priceCurrency":"EUR"}),
}
prepared = prepare_rows([example, different_model, different_currency], role="dev", fit_idf=fit_idf)
feature_values = [dict(zip(FEATURE_NAMES_42, row["x42"])) for row in prepared]
columns = ("price_comparable", "title_compact_exact", "native_model_compact_exact",
           "native_model_same_letters_different_digits")
print(text["feature_width"], ":", len(feature_values[0]))
print(" | ".join([text["case"], *[text["features"][key] for key in columns]]))
for case, row, values in zip(("same_model", "different_model", "different_currency"), prepared, feature_values):
    print(" | ".join([text["cases"][case], *[f"{values[key]:.0f}" for key in columns]]))
    assert len(values)==42 and "label" not in row
assert feature_values[0]["native_model_compact_exact"] == 1
assert feature_values[1]["native_model_compact_exact"] == 0
assert feature_values[1]["native_model_same_letters_different_digits"] == 1
assert feature_values[2]["price_comparable"] == 0


合成例子的特征数量 : 42
合成输入 | 价格可比较 | 紧凑标题完全一致 | 紧凑型号完全一致 | 型号字母相同、数字不同
型号相同 | 1 | 1 | 1 | 0
型号数字不同 | 1 | 0 | 0 | 1
币种不同 | 0 | 1 | 1 | 0


### 输入用途边界校验

给评价输入添加目标标签会触发 `AdmissionError`，仅由训练数据拟合的 IDF 状态保持不变。这验证的是当前输入规则，无法据此确认历史数据使用是否存在泄漏。


In [5]:
from product_matching.errors import AdmissionError
before = fit_idf.payload()
try:
    prepare_rows([{**example, "label":1}], role="evaluation", fit_idf=fit_idf)
except AdmissionError:
    print(text["label_rejected"])
else:
    raise AssertionError(text["label_accepted"])
assert before == fit_idf.payload()
print(text["idf_unchanged"])


评价输入拒绝了目标标签。
仅由训练数据拟合的 IDF 状态没有改变。


## 4. AB/BA 方向与三种子概率集成

A 和 B 表示两条商品记录。RoBERTa-base 分别读取 AB 和 BA 两种商品顺序。64 维投影与 42 项比较特征拼接，再经过一个 64 维 GELU 层、dropout 和标量输出，得到 logit，即经 sigmoid 转成概率之前的原始分数。每个种子先把 AB/BA 的 logit 取均值，再用 sigmoid 转成概率；三个种子的概率最后等权平均。

C+ 按六轮调度训练，开发数据选择共同第 4 轮检查点用于集成。集成保留全部三个种子。训练时编码器使用 BF16 自动混合精度，参数和分类头使用 FP32；推断使用 FP32。固定配置见[模型说明](../docs/zh-CN/MODEL_CARD.md)。

下一单元使用**编写的概率 0.9、0.3 和 0.3**讲解计算。它们不是 RoBERTa 输出，也不是真实种子的实验结果。


In [6]:
from product_matching.metrics import mean_scores
invented = {seed:[{"pair_id":"synthetic:acme", "score":value}]
            for seed,value in zip((42,43,44),(.9,.3,.3))}
mean = mean_scores(["synthetic:acme"], invented)[0]["score"]
print(text["mean"], ":", mean)
print(text["seeds"], ":", facts["model"]["seeds"], ";", text["epoch"], ":", facts["model"]["epoch"])
assert mean == .5


编写的示例概率：等权均值 : 0.5
实验随机种子 : [42, 43, 44] ; 共同训练轮数 : 4


## 5. 排序指标、阈值与混淆计数

AP 根据排序中精确率和召回率的变化计算。并列分数作为一组处理，不会因为其中某条记录排在前面就随意给它额外优势。下面编写的例子里，最高分并列的两条包含一个正例和一个负例，AP 为 0.583333；这不是分类准确率。

真实实验在校准数据上按 F1 选择阈值；并列时先看精确率，再选更高阈值。评价阶段不重新选择阈值。合成 AP 例子之后，代码读取 C+ **已保存的聚合混淆计数**，按 `2 × TP / (2 × TP + FP + FN)` 独立复算 F1。

TP 是找到的真实匹配，FP 是误判为匹配，FN 是漏掉的匹配，TN 是正确排除的不匹配。修改阈值可能改变这些计数和 F1，但不会改变原有排序的 AP。


In [7]:
from product_matching.metrics import average_precision
toy = [{"pair_id":"synthetic:a", "label":1, "score":.9},
       {"pair_id":"synthetic:b", "label":0, "score":.9},
       {"pair_id":"synthetic:c", "label":1, "score":.5}]
print(f'{text["toy_ap"]}: {average_precision(toy):.6f}')
print(text["threshold"], ":", cplus["threshold"])
confusion = cplus["confusion"]
tp,fp,fn,tn = (confusion[k] for k in ("tp","fp","fn","tn"))
f1_from_counts = 2*tp/(2*tp+fp+fn)
assert math.isclose(f1_from_counts, cplus["f1"], rel_tol=0, abs_tol=1e-14)
print(f'{text["confusion"]}: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
print(f'{text["f1_recovered"]}: {f1_from_counts:.6f}')


编写的示例分数：并列情况下的 AP: 0.583333
已保存的 C+ 校准阈值 : 0.3947772259513537
已保存的聚合计数: TP=165, FP=11, FN=25, TN=354
由已保存计数复算的 F1: 0.901639


## 6. 复现范围与执行命令

| 操作 | 已有证据覆盖什么 |
| --- | --- |
| 公开轻量检查与这份 Notebook | 已保存聚合结果的算术、输入规则和合成例子；无需商品数据集或权重 |
| 现有权重回放 | 使用本地提供的原始输入与产物，在 Windows RTX 3060 笔记本上复现保存预测，差值为零 |
| 原始来源角色重建 | 四份业务输入文件及成员顺序与保存的原文件一致；恢复原有成员 |
| 从头重建整个实验 | 新的完整训练尚未从头到尾验证 |

这些单元覆盖第一行。现有权重回放加载已经训练好的模型，不会重新训练，也不是对独立新样本的评价。角色重建恢复历史输入，不能补全历史接触记录。公开仓库不分发商品记录或训练权重。

按照 [README 快速开始](../README.zh-CN.md#快速开始)创建轻量项目环境后，在仓库根目录使用其显式解释器运行：

| 平台 | 公开轻量检查 | Notebook |
| --- | --- | --- |
| macOS / Linux | `.venv/bin/python scripts/public_smoke.py` | `.venv/bin/python scripts/run_notebook.py` |
| Windows | `.\.venv\Scripts\python.exe scripts/public_smoke.py` | `.\.venv\Scripts\python.exe scripts/run_notebook.py` |

使用现有权重实际推断，需要本地提供的商品输入、模型快照和训练权重；推断与从头训练的单独命令，以及详细回放证据，见[复现指南](../docs/zh-CN/REPRODUCIBILITY.md)。


In [8]:
for key in ("max_prediction_difference", "new_training_in_replay", "fresh_training_reproduced"):
    value = facts["reproduction"][key]
    display = text["yes"] if value is True else text["no"] if value is False else value
    print(f'{text["evidence"][key]}: {display}')
heavy = {"torch", "transformers", "sklearn", "sentence_transformers"}
assert not heavy & set(sys.modules)


已记录的回放：最大预测差值: 0.0
回放期间是否重新训练: 否
是否已验证完整重训: 否


## 继续阅读

[数据卡](../docs/zh-CN/DATA_CARD.md)说明字段及全部 42 项比较，[模型说明](../docs/zh-CN/MODEL_CARD.md)给出完整配置，[局限说明](../docs/zh-CN/LIMITATIONS.md)解释固定样本证据能够支持什么结论。[项目卡](../docs/zh-CN/PROJECT_CARD.md)提供简明概览。
